# Look-ahead · Behaviour  `[EVAL]`

**RQ-i on the *behaviours* the rubrics are made of: what does K-turn look-ahead change in what the therapist actually does?** `lookahead/reward` asks the K question of the eight *rubrics*; this family asks it of the oracle-coded **behaviour channels** (the six MI-inconsistent acts, the MITI-coded MI-consistent acts — per therapist turn AND per session), the deterministic **session shape** (length, turn count, turn length, questions per turn, loop degeneracy — judge-invariant), the **selection-level lexical push** each update applied, and the **held-out instruments** the training reward never saw (WAI-SR subscales, patient change talk, the Q2 item profile, the contrast within patient cooperation level). Same optimizer at K=0 vs K=5 (`PTO_LA0` vs `PTO_LA5`, `GRPO_LA0` vs `GRPO_LA5`), persona-paired at every iteration both arms reached, under **both graders side by side** wherever a grader is involved.

**Ported 2026-08-18** from `7_Stats` §4d (`k_paired_channels`, `k_means_channels`, `k_mici_composition` + the four §4d figures — previously rendered per grader under `results/L5/…/7_stats/<judge>/`; the tables now carry a `judge` column, the figures a `_<judge>` suffix) and from the look-ahead paper's promoted generators: `eda_analysis.lookahead.channel_k_frames` (← `k_contrast_headline.py` channels/text part), `eda_analysis.replication` (← `session_shape_stability.py`: shape, length endpoints/contrast, selection table) and `eda_analysis.instruments` (← `held_out_instruments.py`). Statistics unchanged; artifact names drop the script prefix (`k_contrast_headline_channels_pto_primary` → `k_channels_pto_gpt-4o-mini`, `session_shape_stability_shape` → `session_shape`, `held_out_instruments_wai` → `wai_subscales`).

**Conventions (every caption restates them).**
- **Sign: `+ ⇒ K=0 higher`** (K=0 minus K=5) — and on a behaviour channel that is **not a valence**: `+` on a `MICI_*` channel (lower-is-better) means the K=0 policy does *more of a bad thing*; `+` on `mean_turn_len` means only that it writes longer turns. Length metrics are unvalenced.
- **Pairing unit: `persona_id`** — the 96 patient personas recur in every model state (the trainer reshuffles them each iteration, so `file_index` is never a pairing key). The selection table is group-level (no pairing).
- **Iteration 0 = two independent base draws** of the same 96 personas — the noise floor, never dropped.
- **Support: every endpoint is READ OFF THE DATA, per grader** — each arm's rows run to its own last scored iteration, and captions *derive* the support sentence (`constants.support_note`, via the `support()` helper in §0) rather than asserting one, so it disappears when nothing is short. All four arms currently reach iteration 10 under both graders. ⚠ Matched *iteration* is still not matched *budget*: `compute/cost` bills `GRPO_LA5` at 51.205 GPU-h against `GRPO_LA0`'s 27.906, i.e. 51.205 / 27.906 = 1.84×.
- **Holm scope is stated per table** — `7_Stats`-style tables correct across channels WITHIN each (grader, method, iteration, family); the promoted paper tables correct across iterations WITHIN each (grader, method, channel). Same Δ / *dz* / *p*, different `p_holm`.
- **The two graders are never averaged**; oracle-coded channels carry a `judge` column, deterministic text channels are labelled judge-invariant.
- Bootstrap CIs are seeded with `constants.BOOT_SEED`; means / *dz* / *p* / *n* are exact w.r.t. the paper fixture, CI bounds agree to bootstrap noise (count/char-scale channels differ by a few units).

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 60)

import eda_analysis
from eda_analysis import exports, plotting, stats, behavior, lookahead, replication, instruments, reliability
from eda_analysis.constants import BOOT_SEED, DISPLAY_NAMES, set_active_judge, judge_dirname, PRIMARY_JUDGE_TAG

cfg = eda_analysis.EdaConfig(family="lookahead/behaviour", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp figures/_provenance.md (reset just removed the one notebook_setup wrote)

## 0 · Both graders' frames — rubrics, behaviour channels, text metrics  `[EVAL]`
**Purpose.** This family is judge-invariant: it loads every grader in the score lake itself and keeps them apart. `SC` = the rubric `scores_long` per grader (`scores_by_judge`, primary first). The oracle-coded behaviour channels have no `scores_by_judge` twin, so they are loaded **per grader by switching the active judge** (`constants.set_active_judge`) — the channel frame (`channel_scores_long`), the per-conversation wide frame the trajectory figures average (`channels_per_conv`) and the MICI per-conversation detail (`mici_detail_per_conv`) — then the process is left on the primary grader. The deterministic text metrics (`replication.shape_text_metrics` = `behavior.text_metrics` restricted to the four arms, persona attached, the 96-per-cell pairing invariant asserted) are computed once — they are grader-free.

In [ ]:
SC = eda_analysis.scores_by_judge(S)          # {'gpt-4o-mini': scores_long, 'claude-haiku-4-5': scores_long}
JUDGES = list(SC)
PRIMARY, HELDOUT = JUDGES[0], (JUDGES[1] if len(JUDGES) > 1 else None)
assert HELDOUT is not None, "lookahead/behaviour needs the primary AND a held-out judge on disk"
KA = eda_analysis.cross_k_arms(S)             # every arm under the all-arms default (== S.ARMS)
K_ARMS = [a for a in ["PTO_LA0", "PTO_LA5", "GRPO_LA0", "GRPO_LA5"] if a in set(SC[PRIMARY].arm.unique())]
PAL = plotting.arm_palette(sorted(SC[PRIMARY].arm.unique()))
JUDGE_TITLE = {PRIMARY: f"{PRIMARY} (training oracle)", HELDOUT: f"{HELDOUT} (held-out judge)"}
JUDGE_SHORT = {PRIMARY: "training oracle", HELDOUT: "held-out judge"}


def support(frame, **kw) -> str:
    """The support/censoring sentence for a caption, DERIVED from `frame` (constants.support_note).

    Returns "" — not a claim — when every arm in the frame reaches the same iteration, so a caption
    interpolating it says nothing rather than something false once an arm finishes. Leading space so
    it can be dropped at the end of a sentence. Never hardcode "GRPO_LA5 is right-censored": that was
    true only while that arm was short, and it outlived the condition in ~20 rendered captions.
    """
    note = eda_analysis.support_note(frame, **kw)
    return f" {note}" if note else ""


for j, sc in SC.items():
    print(f"{j:>18}: {sc.shape} | arms {sorted(sc.arm.unique())} | iters "
          + ", ".join(f"{a}:0..{int(sc[sc.arm == a].iteration.max())}" for a in K_ARMS))

# ── behaviour channels PER GRADER (module-level active judge; restored to the primary afterwards) ──
TAGS = {judge_dirname(t): t for t in reliability.second_judge_tags()}
TAGS[judge_dirname(PRIMARY_JUDGE_TAG)] = ""
CH, CHW, MICI_PC = {}, {}, {}
try:
    for j in JUDGES:
        set_active_judge(TAGS[j], 0)
        CH[j] = behavior.channel_scores_long(KA)          # long: questionnaire = CHANNEL id
        CHW[j] = behavior.channels_per_conv(KA)           # wide, per conversation (persona attached)
        MICI_PC[j] = behavior.mici_detail_per_conv(KA)    # MICI counts + rates per conversation
        print(f"channels[{j}]: {CH[j].shape} | {CH[j].questionnaire.nunique()} channels | arms {sorted(CH[j].arm.unique())}")
finally:
    set_active_judge("", 0)                               # leave the process on the primary grader

# ── deterministic text metrics, once (judge-invariant) ──
TM = replication.shape_text_metrics(KA)                   # per conversation, four arms, persona attached
print("text metrics:", TM.shape, "| arms:", sorted(TM.arm.unique()))
print("derived support note (empty => every arm reaches the same iteration):", repr(support(TM)))

## 1 · The `7_Stats` §4d artifacts — the K contrast on every behaviour channel, both graders  `[EVAL]`
**Purpose.** The tracked RQ-i behaviour tables, unchanged in statistic and name, now with a `judge` column instead of a `<judge>/` folder: `k_paired_channels` (persona-paired K0 − K5 per channel × iteration, **Holm across the channels WITHIN each (grader, method, iteration, family)** — the families are `behavior.BEHAVIOR_CHANNEL_FAMILIES`, named in the `family` column), `k_means_channels` (the un-paired LEVELS the test is computed on) and `k_mici_composition` (the per-session count of each MI-inconsistent act, their sum, the severity global, and each act's SHARE of the session total — the aggregate control behind every per-turn rate claim).

**Why it needs its own section.** A rubric is a weighted summary of behaviours, and a summary can be flat while its components move in opposite directions. Two controls have to pass before a rate gap is a behaviour claim: (1) **the denominator** — every `*_rate` divides by `n_th_turns`, and the arms differ on it in *opposite* directions per method (PTO's K=5 arm takes MORE therapist turns, GRPO's K=5 arm FEWER), so the raw per-session counts are tested as their own family; (2) **the aggregate** — `MICI_BehaviorTotal` is the sum of all six MI-inconsistent acts; if it is unchanged while one component collapses, the policy *substituted* rather than improved. Check both before writing "mitigation" anywhere.

The four figures repeat per grader (`_<judge>` suffix): the per-SESSION composition grid, the over-praise trajectory, the channel forest at the last matched PTO iteration, and the reward-vs-hack cost–benefit frame.

In [ ]:
KPC_ALL, KMC_ALL, COMP_ALL, FAMS = [], [], [], {}
for j in JUDGES:
    ch = CH[j]
    present = set(ch.questionnaire.unique())
    FAMILIES = {f: [c for c in cs if c in present] for f, cs in behavior.BEHAVIOR_CHANNEL_FAMILIES.items()}
    FAMILIES = {f: cs for f, cs in FAMILIES.items() if cs}
    FAMS[j] = FAMILIES
    # ── the TEST: per family, so each Holm correction covers one hypothesis set ──
    frames = []
    for fam, chans in FAMILIES.items():
        for m in ["PTO", "GRPO"]:
            C = stats.paired_k_comparison(ch, m, metrics=chans)
            if not C.empty:
                frames.append(C.assign(family=fam))
    if frames:
        KPC_ALL.append(pd.concat(frames, ignore_index=True).assign(judge=j))
    # ── the LEVELS the test is computed on ──
    ALL_CH = [c for cs in FAMILIES.values() for c in cs]
    KM = [M for M in (stats.k_means_by_iter(ch, m, metrics=ALL_CH) for m in ["PTO", "GRPO"]) if not M.empty]
    if KM:
        KMC_ALL.append(pd.concat(KM, ignore_index=True).assign(judge=j))
    # ── THE AGGREGATE CHECK: is the MI-inconsistent total lower, or just recomposed? ──
    MP = MICI_PC[j]
    if not MP.empty:
        comp_cols = ["MICI_BehaviorTotal"] + list(behavior._MICI_RATE_BEHAVIORS) + ["MICI_Severity"]
        COMP = (MP.groupby(["arm", "method", "K", "iteration"], observed=True)
                [[c for c in comp_cols if c in MP.columns]].mean().reset_index())
        for b in behavior._MICI_RATE_BEHAVIORS:          # share of the session's MI-inconsistent acts per behaviour
            if b in COMP.columns:
                COMP[f"{b}_share"] = COMP[b] / COMP["MICI_BehaviorTotal"].where(COMP["MICI_BehaviorTotal"] > 0)
        COMP_ALL.append(COMP.assign(judge=j))

KPC = pd.concat(KPC_ALL, ignore_index=True)
KPC_VIEW = KPC[["judge", "method", "family", "iteration", "metric", "n", "mean_delta", "dz", "p", "p_holm"]].round(4)
print("=== K0 - K5 on the behaviour channels (+ => the K=0 policy does MORE of it), both graders ===")
display(KPC_VIEW[KPC_VIEW.metric.isin(["MICI_BehaviorTotal", "MICI_OverPraise", "MICI_AdviseNoPermission", "MICI_Severity"])])
exports.save_table(KPC_VIEW, "k_paired_channels", caption=(
    "RQ-i on the BEHAVIOUR channels, both graders (column `judge`: gpt-4o-mini = training oracle, claude-haiku-4-5 = "
    "held-out judge; the two are never averaged): K0 - K5 within each method at every matched iteration (0 = the two "
    "independent base draws), persona-paired (persona_id, n = 96) Wilcoxon + Cohen's dz + Holm. + => the K=0 policy does "
    "MORE of the channel — NOT 'better': these are behaviour counts, so read valence off constants.LOWER_IS_BETTER (every "
    "MICI_* channel is higher = worse) and treat mean_turn_len / conv_len / n_th_turns as unvalenced. Holm scope: "
    "corrected across channels WITHIN each (judge, method, iteration, family); the `family` column names the set "
    "(behavior.BEHAVIOR_CHANNEL_FAMILIES). Read the per-SESSION families before the per-TURN ones: every rate divides "
    "by n_th_turns, which itself differs between the arms. Each method's rows run to its own last MATCHED iteration — "
    "read that endpoint off the `iteration` column, per grader."
    + support(KPC, arm_col="method", label=False, subject="no later matched iteration in this frame")
    + " Same statistic as the retired results/L5/tables/7_stats/<judge>/k_paired_channels (one table per grader). "
    f"Bootstrap-free (Wilcoxon + dz)."))

KMC = pd.concat(KMC_ALL, ignore_index=True)
KMC = KMC[["judge"] + [c for c in KMC.columns if c != "judge"]].round(4)
exports.save_table(KMC, "k_means_channels", caption=(
    "RQ-i behaviour-channel LEVELS, both graders (column `judge`): K=0 vs K=5 arm means per method x channel x iteration "
    "with the unpaired delta (+ => K0 higher) and each side's n (96 conversations per cell). The companion to "
    "k_paired_channels — read dz/p there, never off this table. The `iteration` column names each method's last matched "
    "point, per grader."
    + ("" if KMC[["mean_K5", "delta"]].notna().to_numpy().all() else
       " An iteration one arm never reached keeps mean_K5/delta NaN; there are such rows in this render.")
    + support(KMC, arm_col="method", label=False, subject="no later matched iteration in this frame")
    + " Every per-turn value here also appears in the "
    "arms/questionnaires mici_behavior_by_iter / miti_detail_by_iter / session_shape_by_iter tables under the same "
    "column name, by construction. Same statistic as the retired results/L5/tables/7_stats/<judge>/k_means_channels."))

COMP = pd.concat(COMP_ALL, ignore_index=True)
COMP = COMP[["judge"] + [c for c in COMP.columns if c != "judge"]].round(4)
display(COMP[COMP.judge == PRIMARY])
exports.save_table(COMP, "k_mici_composition", caption=(
    "MI-inconsistent behaviour COMPOSITION per (grader, arm, iteration) — column `judge` names the coder (gpt-4o-mini = "
    "training oracle, claude-haiku-4-5 = held-out judge): the per-session count of each of the 6 behaviours, their sum "
    "(MICI_BehaviorTotal), the severity global, and each behaviour's SHARE of the session total. The aggregate control "
    "behind every per-turn rate claim: a component collapsing while the total holds means the policy SUBSTITUTED one "
    "violation for another, not that it committed fewer. Counts are per session and so carry no n_th_turns denominator; "
    "the share columns are denominator-free altogether. Arm means over 96 conversations; each arm's rows run to its own "
    "last scored iteration under that coder (`iteration` column)."
    + support(COMP, subject="no later scored state under this coder")
    + " Same statistic as the retired results/L5/tables/7_stats/<judge>/k_mici_composition."))

In [ ]:
# ── FIGURES, one set per grader ─────────────────────────────────────────────────────────────
# PRIMARY = the per-SESSION count, because that is the quantity with no moving denominator (the arms
# differ in therapist-turn count). The per-turn rate is kept as the secondary trajectory only.
HACK, HACK_RATE, AGG, ADV, SEV = "MICI_OverPraise", "MICI_OverPraise_rate", "MICI_BehaviorTotal", "MICI_AdviseNoPermission", "MICI_Severity"

def _c(frame, method, it, metric):
    """One paired contrast as text, read off the table (never eyeballed off the figure)."""
    r = frame[(frame.method == method) & (frame.iteration == it) & (frame.metric == metric)]
    if r.empty:
        return "n/a"
    r = r.iloc[0]
    return f"delta {r.mean_delta:+.2f} (dz {r.dz:+.2f}, Holm p {r.p_holm:.3f})"

ONSET, LAST = {}, {}
for j in JUDGES:
    kpc = KPC[KPC.judge == j]
    # The onset is READ OFF THE TEST, never eyeballed: first iteration at which the paired contrast on
    # the hack channel clears Holm within its own family.
    sig = kpc[(kpc.method == "PTO") & (kpc.metric == HACK) & (kpc.p_holm < 0.05)]
    onset = int(sig.iteration.min()) if not sig.empty else None
    ONSET[j] = onset
    last = int(kpc[kpc.method == "PTO"].iteration.max()); glast = int(kpc[kpc.method == "GRPO"].iteration.max())
    LAST[j] = (last, glast)
    print(f"[{j}] PTO divergence onset on {HACK} -> {onset} | last matched iteration PTO {last}, GRPO {glast}")
    ENDPT = (f"At the matched endpoints the persona-paired K0-K5 contrasts (k_paired_channels, per-session family; + => "
             f"K=0 does MORE) read: PTO iteration {last} — over-praise {_c(kpc, 'PTO', last, HACK)}, advice w/o permission "
             f"{_c(kpc, 'PTO', last, ADV)}, MI-inconsistent total {_c(kpc, 'PTO', last, AGG)}, severity global "
             f"{_c(kpc, 'PTO', last, SEV)}; GRPO iteration {glast} — over-praise {_c(kpc, 'GRPO', glast, HACK)}, advice "
             f"{_c(kpc, 'GRPO', glast, ADV)}, total {_c(kpc, 'GRPO', glast, AGG)}, severity {_c(kpc, 'GRPO', glast, SEV)}.")

    fig = plotting.k_channel_trajectory_grid(
        CHW[j], [HACK, ADV, AGG, SEV],
        arms=K_ARMS, onset={HACK: onset} if onset else None,
        suptitle=f"Does look-ahead reduce MI-inconsistency, or recompose it? (K=0 solid · K=5 dashed) — coder: {JUDGE_TITLE[j]}")
    exports.save_fig(fig, f"k_mici_composition_grid_{j}", caption=(
        f"The composition check, in per-SESSION counts so no denominator is involved — MICI coder = {JUDGE_TITLE[j]}. "
        "Panels: over-praise per session (top left, the hacked channel), advice-without-permission per session (top right), "
        "the TOTAL number of MI-inconsistent acts per session (bottom left) and the MICI severity global (bottom right); "
        "arm means over 96 personas; higher = worse on every panel; each line runs to that arm's last scored iteration "
        f"under this coder.{support(CHW[j], subject='no later scored state under this coder')} {ENDPT} Read the total "
        "against the over-praise panel: where the over-praise gap is large but "
        "the total gap is small (or the other components rise), look-ahead SUBSTITUTED one violation for another rather "
        "than removing violations; where the total gap is large too, K=0 commits more violations outright. Shading marks "
        "the divergence onset on over-praise, read off k_paired_channels (first PTO iteration clearing Holm within its "
        "family) — " + (f"iteration {onset}." if onset else "none cleared Holm under this coder, so no shading.")
        + " Same figure as the retired results/L5/figures/7_stats/<judge>/k_mici_composition_grid."))
    plt.show()

    fig = plotting.k_channel_trajectory(CHW[j], HACK_RATE, arms=K_ARMS, annotate_from=onset,
                                        title=f"The over-praise channel, per policy iteration — coder: {JUDGE_TITLE[j]}")
    exports.save_fig(fig, f"k_overpraise_trajectory_{j}", caption=(
        f"Over-praise per therapist turn, every arm, per iteration (K=0 solid, K=5 dashed) — MICI coder = {JUDGE_TITLE[j]}. "
        "The channel that carries the MICI rise in both K=0 arms; from the mid iterations on the K=5 arms sit far below "
        f"their K=0 siblings under both coders (read the levels off k_means_channels).{support(CHW[j], subject='no later scored state under this coder')} "
        f"Per-turn rate at the endpoints (k_paired_channels, per-turn family): PTO "
        f"iteration {last} {_c(kpc, 'PTO', last, HACK_RATE)}; GRPO iteration {glast} {_c(kpc, 'GRPO', glast, HACK_RATE)}. "
        "Shading marks the divergence onset read off k_paired_channels. Higher = worse. Read with k_mici_composition_grid "
        "before calling this a reduction of MI-inconsistency: the per-turn rate has a moving denominator (n_th_turns "
        "differs between the arms) and the per-session TOTAL is the aggregate control. Arm means over 96 personas. Same "
        "figure as the retired results/L5/figures/7_stats/<judge>/k_overpraise_trajectory."))
    plt.show()

    fig = plotting.k_channel_forest(
        kpc, iteration=last, method="PTO",
        title=f"PTO K=0 - K=5 on every behaviour channel at iteration {last} — coder: {JUDGE_TITLE[j]}",
        caption="Sign convention + = the K=0 policy does MORE of the channel. Colour is VALENCE, not direction: red = a "
                "MI-inconsistent behaviour K=0 does more of, green = one K=5 does more of, grey = unvalenced session "
                "shape. Red AND green bars both being large would mean the violations were swapped, not removed. "
                "Hollow = did not clear Holm within its (iteration, family) set.")
    exports.save_fig(fig, f"k_channel_forest_{j}", caption=(
        f"Every behaviour channel's persona-paired dz (n = 96) at the last matched PTO iteration ({last}) — coder = "
        f"{JUDGE_TITLE[j]} — the trade-off in one frame. + => the K=0 policy does more of it. Colour encodes valence (red = "
        "MI-inconsistent and higher under K=0; green = MI-inconsistent and higher under K=5; grey = unvalenced session "
        "shape); hollow = did not clear Holm within its (iteration, family) set. Read it as: which violations K=0 commits "
        "more of (red), which K=5 commits more of (green), and whether the per-session TOTAL bar sits near zero (a swap) or "
        f"with the red bars (K=0 commits more outright) — at iteration {last}: over-praise/session {_c(kpc, 'PTO', last, HACK)}, "
        f"advice w/o permission/session {_c(kpc, 'PTO', last, ADV)}, total/session {_c(kpc, 'PTO', last, AGG)}. Same "
        "figure as the retired results/L5/figures/7_stats/<judge>/k_channel_forest."))
    plt.show()

    # Reward vs hack in one frame — needs BOTH contrasts, so run the rubric contrast under the same grader.
    RUB = pd.concat([C for C in (stats.paired_k_comparison(SC[j], m) for m in ["PTO", "GRPO"]) if not C.empty],
                    ignore_index=True)
    fig = plotting.k_cost_benefit(pd.concat([RUB, kpc.drop(columns=["judge"])], ignore_index=True),
                                  reward_metric=cfg.focus_metric, hack_channel=HACK, method="PTO", control_channel=AGG,
                                  title=f"PTO, K=0 - K=5 as dz: reward vs the hacked channel vs its aggregate — grader: {JUDGE_TITLE[j]}")
    exports.save_fig(fig, f"k_cost_benefit_{j}", caption=(
        f"The RQ-i answer in one frame, grader = {JUDGE_TITLE[j]}: the paired K contrast on the REWARD the run optimised "
        f"({DISPLAY_NAMES.get(cfg.focus_metric, cfg.focus_metric)}), on the CHANNEL it was hacked through ({HACK}, per-session "
        f"count) and on the AGGREGATE that channel belongs to ({AGG}) — all as dz, all + => K=0 higher, persona-paired "
        f"(n = 96), PTO only. At iteration {last}: reward {_c(RUB, 'PTO', last, cfg.focus_metric)} (Holm across rubrics), "
        f"hacked channel {_c(kpc, 'PTO', last, HACK)}, aggregate {_c(kpc, 'PTO', last, AGG)} (Holm within the per-session "
        "family). Where the reward line stays small while the channel line is large, the two arms are near-equivalent by "
        "the objective and far apart in WHICH violation they commit; the aggregate line says whether that is a swap (near "
        "zero) or fewer violations under K=5 (positive). Iteration 0 is the two independently generated base models and is "
        "the noise floor. Circled markers cleared Holm within their own family. Same figure as the retired "
        "results/L5/figures/7_stats/<judge>/k_cost_benefit."))
    plt.show()

In [ ]:
# ── GRPO-only channel forest — the endpoint trade-off for the GRPO-scoped write-up ──────────
# The k_channel_forest_<judge> above is the PTO forest (method="PTO" at the last matched PTO
# iteration). The GRPO-scoped paper needs the same frame at ITS endpoint, under the same
# conventions (sign + = K=0 does MORE; colour = valence; hollow = failed Holm).
for j in JUDGES:
    kpc = KPC[KPC.judge == j]
    glast = int(kpc[kpc.method == "GRPO"].iteration.max())
    fig = plotting.k_channel_forest(
        kpc, iteration=glast, method="GRPO",
        title=f"GRPO K=0 - K=5 on every behaviour channel at iteration {glast} — coder: {JUDGE_TITLE[j]}",
        caption="Sign convention + = the K=0 policy does MORE of the channel. Colour is VALENCE, not direction: red = a "
                "MI-inconsistent behaviour K=0 does more of, green = one K=5 does more of, grey = unvalenced session "
                "shape. Red AND green bars both being large would mean the violations were swapped, not removed. "
                "Hollow = did not clear Holm within its (iteration, family) set.")
    exports.save_fig(fig, f"k_channel_forest_grpo_{j}", caption=(
        f"Every behaviour channel's persona-paired dz (n = 96) at the last matched GRPO iteration ({glast}) — coder = "
        f"{JUDGE_TITLE[j]}. + => the K=0 policy does more of it. Colour encodes valence (red = MI-inconsistent and "
        "higher under K=0; green = MI-inconsistent and higher under K=5; grey = unvalenced session shape); hollow = "
        "did not clear Holm within its (iteration, family) set. Companion of k_channel_forest_<judge>, which draws "
        "the PTO forest. Read it as: which violations K=0 commits more of (red — over-praise dominates), which K=5 "
        "commits more of (green), and whether a large red AND green pair would mean recomposition rather than "
        "removal."))
    plt.show()


### 1b · The same claim without a judge — lexical over-praise beside the two rated rates  `[EVAL]`
**Purpose.** Everything above codes over-praise with an **LLM**, which is the one kind of evidence a sceptical reader will not accept as proof of a *reward-hacking* claim *about* an LLM judge — the coder and the graded policy come from the same family of model. There is a measurement that owes the judge nothing: `lex_overpraise_marker_rate`, a deterministic keyword regex (`constants.RE_EFFUSIVE`: "I'm so proud", "proud of you", "beacon", "you got this", "warrior", …) run over the therapist turns of the transcript. It is already tracked next to the rated rate — `results/arms/validity/tables/<judge>/overpraise_crosscheck.md` puts the two in adjacent columns, and `results/arms/validity/figures/<judge>/overpraise_crosscheck.png` already plots them against each other, per grader. What is **not** anywhere is the judge-free column beside **both** graders' rated columns on one x-axis, which is what makes "the same story under three independent measurements" legible at a glance. That, and only that, is what this section adds. This section puts the three side by side on one x-axis, all four arms: **(a)** judge-free, **(b)** rated by the training oracle, **(c)** rated by the held-out judge.

**What it shows.** The judge-free panel reproduces the graded story — same monotone ordering of the four arms at the endpoint, the same K=0-diverges/K=5-stays-flat shape — with no LLM anywhere in its numerator. That is the point: the over-praise finding does not depend on trusting a model to count over-praise.

**The traps.**
- ⚠ **The `<judge>/` in the source path is a filing convention, not a dependency.** `arms/*` is a per-judge FAMILY — *every* artifact under that top is written once per grader, judge-invariant ones included. `lex_overpraise_marker_rate` is computed from the transcripts, so the two copies of `overpraise_crosscheck.md` must carry the **identical** column. The cell **asserts** that (byte-identical, not merely close) rather than claiming it, and the caption reports the measured max difference. This family is judge-invariant and writes no `<judge>/` level, so it can publish the column as one line per arm.
- ⚠ **Three panels, three independent y-axes, no level comparison across them.** Never across the two graders (the held-out judge's offset is model-dependent), and not between (a) and (b)/(c) either: the denominator is the same (**per therapist turn**, `n_th_turns`) but the *numerator* is not — (a) is an **incidence** rate (fraction of therapist turns containing ≥ 1 marker, bounded 0–1), (b)/(c) are **count** rates (coded over-praise acts per therapist turn, unbounded). Shape and ordering transfer; levels do not.
- ⚠ **The regex is a narrow keyword list**, kept in `METRICS_REFERENCE` §3c as a *directional* sanity-check and deliberately excluded from the headline behaviour metrics. Its absolute level is a **lower bound** — it cannot see the paraphrases the coder catches. It licenses the ordering and the trajectory, not the magnitude.
- The frame is built from what §0 already loaded (`TM` × `MICI_PC[j]`, inner-joined on `(arm, iteration, file_index)`) along the same builder path as `behavior.overpraise_crosscheck`, so it reproduces `arms/validity`'s rendered table cell-for-cell; the figure and `overpraise_judgefree_data` are driven off the *same* frame object.

In [ ]:
# ── 1b · JUDGE-FREE over-praise beside the graded over-praise ───────────────────────────────
# Built from what §0 already loaded: TM (deterministic transcript metrics, no grader) inner-joined
# per grader with MICI_PC[j] on (arm, iteration, file_index) — the SAME builder path as
# behavior.overpraise_crosscheck, so this reproduces results/arms/validity/tables/<judge>/
# overpraise_crosscheck.md cell-for-cell (both columns, both graders).
LEX_OP, RATED_OP = "lex_overpraise_marker_rate", "MICI_OverPraiseRate"
_XK = ["arm", "iteration", "file_index"]

_by_judge = {}
for j in JUDGES:
    _m = TM.merge(MICI_PC[j][_XK + [RATED_OP]], on=_XK, how="inner")
    _g = _m.groupby(["arm", "method", "K", "iteration"], observed=True)
    _by_judge[j] = (_g[[LEX_OP, RATED_OP]].mean().join(_g.size().rename("n_conv"))
                    .reset_index().sort_values(["arm", "iteration"]).reset_index(drop=True))

# ⚠ THE CHECK the whole figure rests on. The lexical rate is computed from the transcripts, so the
# graders' copies must be IDENTICAL, not merely close — ASSERT it rather than assert it in prose.
# (Its source file sits under an arms/validity/tables/<judge>/ path only because arms/* is a
# per-judge FAMILY; the value is not grader-dependent.)
_lx = {j: d.set_index(["arm", "iteration"])[LEX_OP].sort_index() for j, d in _by_judge.items()}
LEX_MAXDIFF = float(max((_lx[j] - _lx[PRIMARY]).abs().max() for j in JUDGES))
# NOT an assert: the only way these can differ is a MICI coverage mismatch between graders (a
# partially-landed judge sweep - the case filter_complete_cells exists for next door). Aborting
# would take the WHOLE lookahead/behaviour render down over one panel, so degrade instead and say
# so loudly in the caption. (.equals also fails on a bare index mismatch, which would print a NaN.)
LEX_INVARIANT = all(_lx[j].equals(_lx[PRIMARY]) for j in JUDGES)
if not LEX_INVARIANT:
    print(f"WARNING: {LEX_OP} is supposed to be judge-invariant but differs across graders "
          f"(max |diff| {LEX_MAXDIFF:.3e}) - panel (a) is the PRIMARY grader's join only. "
          f"Most likely a MICI coverage mismatch; check multijudge_coverage.")
print(f"judge-free column identical across {len(JUDGES)} graders over {len(_lx[PRIMARY])} (arm, iteration) "
      f"cells: max |diff| = {LEX_MAXDIFF:.1e} | conversations per cell = "
      f"{sorted(_by_judge[PRIMARY].n_conv.unique())}")

RATED_COL = {j: f"{RATED_OP}_{j}" for j in JUDGES}
OP = _by_judge[PRIMARY][["arm", "method", "K", "iteration", "n_conv", LEX_OP]].copy()
for j in JUDGES:
    OP = OP.merge(_by_judge[j][["arm", "iteration", RATED_OP]].rename(columns={RATED_OP: RATED_COL[j]}),
                  on=["arm", "iteration"], how="left")
OP = OP.sort_values(["arm", "iteration"]).reset_index(drop=True)
display(OP[OP.iteration == OP.iteration.max()].round(3))

# ── the three panels: one x-axis, INDEPENDENT y-axes (different numerators, and two graders) ──
_PANELS = [(LEX_OP,
            "(a) JUDGE-FREE — lexical over-praise marker\n(regex over the transcripts; no LLM involved)",
            "therapist turns carrying an effusive marker\n(fraction of therapist turns)")]
_PANELS += [(RATED_COL[j], f"({'bcdefgh'[i]}) ORACLE-RATED — {JUDGE_TITLE[j]}",
             "coded over-praise acts\nper therapist turn") for i, j in enumerate(JUDGES)]

fig, axes = plt.subplots(1, len(_PANELS), figsize=(5.2 * len(_PANELS), 4.1), sharex=True)
axes = np.atleast_1d(axes)
_mid = len(_PANELS) // 2
for i, (ax, (col, ttl, ylab)) in enumerate(zip(axes, _PANELS)):
    plotting.k_channel_trajectory(OP, col, arms=K_ARMS, ax=ax, palette=PAL)
    ax.set_title(ttl, fontsize=9)
    # ⚠ k_channel_trajectory's _unit() keys off the column NAME and cannot see that
    # MICI_OverPraiseRate is a per-TURN rate — so name the denominator explicitly on every panel.
    ax.set_ylabel(ylab, fontsize=8)
    ax.set_xlabel("training iteration (policy that generated the conversations)" if i == _mid else "")
    if ax.get_legend():
        ax.get_legend().remove()
_h, _l = axes[0].get_legend_handles_labels()
fig.legend(_h, _l, loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=len(_l), frameon=False, fontsize=8)
fig.suptitle("Over-praise: the judge-free measurement beside the two graded ones (K=0 solid · K=5 dashed)",
             y=1.10, fontweight="bold")
fig.tight_layout()

# ── does the judge-free panel tell the same story? quantify it; never eyeball it off the figure ──
_END = int(OP.iteration.max())
_E = OP[OP.iteration == _END]
_RHO = {j: float(OP[[LEX_OP, RATED_COL[j]]].corr(method="spearman").iloc[0, 1]) for j in JUDGES}
_ORDERS = {ttl[:3]: tuple(_E.sort_values(col, ascending=False).arm) for col, ttl, _ in _PANELS}
_ORD_AGREE = len(set(_ORDERS.values())) == 1


def _endpt(col):
    d = _E.set_index("arm")[col]
    return ", ".join(f"{a} {d[a]:.3f}" for a in K_ARMS if a in d.index)


print(f"Spearman rho(judge-free, rated) over {len(OP)} (arm, iteration) cells: "
      + ", ".join(f"{j} {_RHO[j]:+.3f}" for j in JUDGES))
print(f"endpoint (iteration {_END}) arm order, worst first — identical across all three panels: {_ORD_AGREE}")
for k, v in _ORDERS.items():
    print(f"   {k} {list(v)}")

exports.save_fig(fig, "overpraise_judgefree", caption=(
    "**The JUDGE-FREE over-praise evidence beside the graded evidence, all four arms** (K=0 solid + circle, "
    "K=5 dashed + square; each point is the arm mean over the 96 conversations of that (arm, iteration) cell). "
    "(a) `lex_overpraise_marker_rate` — a DETERMINISTIC keyword regex (constants.RE_EFFUSIVE: \"I'm so proud\", "
    "\"proud of you\", \"beacon\", \"you got this\", \"warrior\", \"shining\", ...) applied to the therapist turns "
    "of the transcript. No LLM appears anywhere in this quantity, so the panel is IDENTICAL under both graders "
    f"— verified in this render, not asserted: the two graders' copies of the column agree exactly (max |diff| = "
    f"{LEX_MAXDIFF:.1e} over all {len(OP)} (arm, iteration) cells, 96 conversations each). (b) the ORACLE-RATED "
    f"rate under {JUDGE_TITLE[PRIMARY]} and (c) under {JUDGE_TITLE[HELDOUT]} — `MICI_OverPraiseRate`, the MICI "
    "coder's over-praise count over the same therapist-turn denominator. "
    "⚠ DENOMINATOR: all three panels are PER THERAPIST TURN (n_th_turns), but the NUMERATORS differ — (a) is "
    "an INCIDENCE rate (fraction of therapist turns containing at least one marker, bounded 0-1), (b)/(c) are "
    "COUNT rates (coded acts per therapist turn, unbounded). "
    "⚠ THE THREE y-AXES ARE INDEPENDENT and levels are never compared across panels: not across the two "
    "graders (the held-out judge carries a model-dependent level offset from the training oracle, so only "
    "contrasts and orderings travel) and not between (a) and (b)/(c) either (different numerators). What is "
    "comparable is SHAPE and ORDERING, and they agree: Spearman rho between the judge-free series and the rated "
    f"one is " + ", ".join(f"{_RHO[j]:+.3f} ({JUDGE_SHORT[j]})" for j in JUDGES) + f" across all {len(OP)} cells, "
    f"and at iteration {_END} the four arms rank in the "
    + ("SAME order (worst first: " + ", ".join(_ORDERS[list(_ORDERS)[0]]) + ") in all three panels"
       if _ORD_AGREE else "orders given by overpraise_judgefree_data, which are NOT identical across the panels")
    + f". Endpoint values at iteration {_END} — judge-free: {_endpt(LEX_OP)}; "
    + "; ".join(f"rated by {JUDGE_SHORT[j]}: {_endpt(RATED_COL[j])}" for j in JUDGES) + ". "
    "⚠ PATH SUBTLETY: the judge-free column is published at "
    "results/arms/validity/tables/<judge>/overpraise_crosscheck.md, i.e. under a <judge>/ leaf — but ONLY because "
    "arms/* is a per-judge FAMILY whose every artifact is written once per grader. The VALUE is not "
    "grader-dependent, which is exactly what the equality check above establishes; this family is "
    "judge-invariant and writes no <judge>/ level, so it publishes the column as one line per arm. "
    "⚠ The regex is a narrow keyword list (METRICS_REFERENCE section 3c keeps it as a DIRECTIONAL "
    "sanity-check, deliberately outside the headline behaviour metrics): its absolute level is a LOWER BOUND on "
    "over-praise, since it cannot see the paraphrases the coder catches. It licenses the ordering and the "
    "trajectory, not the magnitude. Higher = worse on every panel. Iteration 0 = each arm's own base draw."
    + support(OP, subject="no later scored state in this frame")
    + " Numbers: overpraise_judgefree_data (the frame this figure is drawn from). Same builder path as "
    "behavior.overpraise_crosscheck, so both rated columns reproduce arms/validity's rendered "
    "overpraise_crosscheck.md cell-for-cell."))
plt.show()

exports.save_table(OP.round(4), "overpraise_judgefree_data", caption=(
    "The frame behind overpraise_judgefree, per (arm, iteration): `lex_overpraise_marker_rate` = the JUDGE-FREE "
    "lexical over-praise incidence (fraction of therapist turns matching constants.RE_EFFUSIVE; deterministic, "
    "computed from the transcripts, so it appears ONCE — it is identical under both graders, checked exactly in "
    f"this render: max |diff| = {LEX_MAXDIFF:.1e} across the graders over all {len(OP)} cells), and one "
    "`MICI_OverPraiseRate_<judge>` column per grader = that coder's over-praise COUNT per therapist turn. "
    "n_conv = conversations averaged per cell (96). ⚠ Same denominator (therapist turns), different "
    "numerators — the lexical column is an incidence rate bounded 0-1, the rated columns are unbounded count "
    "rates; and the two rated columns are two graders, never averaged and never compared on level. Higher = "
    "worse in every column. The rated columns reproduce results/arms/validity/tables/<judge>/"
    "overpraise_crosscheck.md exactly (same builder: behavior.text_metrics x behavior.mici_detail_per_conv, "
    "inner-joined on (arm, iteration, file_index)); that table lives under a <judge>/ leaf only because arms/* "
    "is a per-judge family."
    + support(OP, subject="no later scored state in this frame")))

In [ ]:
# ── 1b-ii · GRPO-only companion of the figure above ─────────────────────────────────────────
# Same frame (OP), same three panels, the two PTO arms dropped. Exists so a GRPO-scoped write-up
# (papers/2026_grpo_lookahead_mi (revived 2026-08-27 for ARR Oct)) can show its own subjects in the MAIN text while the four-arm
# version above stays the canonical EDA artifact (and that paper's appendix material). Every
# caveat of the four-arm figure applies unchanged; the numbers are the SAME
# overpraise_judgefree_data table, filtered to the GRPO arms.
_G_ARMS = [a for a in K_ARMS if a.startswith("GRPO")]
fig, axes = plt.subplots(1, len(_PANELS), figsize=(5.2 * len(_PANELS), 4.1), sharex=True)
axes = np.atleast_1d(axes)
for i, (ax, (col, ttl, ylab)) in enumerate(zip(axes, _PANELS)):
    plotting.k_channel_trajectory(OP, col, arms=_G_ARMS, ax=ax, palette=PAL)
    ax.set_title(ttl, fontsize=9)
    ax.set_ylabel(ylab, fontsize=8)
    ax.set_xlabel("training iteration (policy that generated the conversations)" if i == _mid else "")
    if ax.get_legend():
        ax.get_legend().remove()
_h, _l = axes[0].get_legend_handles_labels()
fig.legend(_h, _l, loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=len(_l), frameon=False, fontsize=8)
fig.suptitle("Over-praise, GRPO arms only: the judge-free measurement beside the two graded ones (K=0 solid, K=5 dashed)",
             y=1.10, fontweight="bold")
fig.tight_layout()
_ge = _E.set_index("arm")[LEX_OP]
exports.save_fig(fig, "overpraise_judgefree_grpo", caption=(
    "**GRPO-only companion of `overpraise_judgefree`** — same frame, same three panels ((a) the judge-free "
    "lexical incidence, (b)/(c) the oracle-rated count per therapist turn under each grader), the two PTO arms "
    "dropped so a GRPO-scoped write-up can carry its own subjects in the main text. Every caveat of the "
    "four-arm figure applies unchanged: the three y-axes are INDEPENDENT (incidence vs count numerators; two "
    "graders whose levels never compare), the regex is a directional sanity-check whose absolute level is a "
    "lower bound, and higher = worse everywhere. Numbers: overpraise_judgefree_data filtered to the GRPO arms. "
    f"Endpoint (iteration {_END}) judge-free incidence: "
    + ", ".join(f"{a} {_ge[a]:.3f}" for a in _G_ARMS if a in _ge.index)
    + ". The four-arm version above is the canonical artifact; this one exists for scope, not because the "
    "PTO panels are wrong."))
plt.show()


## 2 · The paper's channel contrast — long form per (grader, method), text channels once  `[EVAL]`
**Purpose.** The promoted `lookahead.channel_k_frames`: the same persona-paired machinery as the rubric tables in `lookahead/reward` §2, applied to every oracle-coded channel (MICI per-turn rates + per-session counts + severity; MITI per-turn rates, counts, R:Q, %CR, %MICO) **per grader**, and to the deterministic text channels (`conv_len`, `n_th_turns`, `mean_turn_len`, `q_per_turn`, `loop`) **once** (judge-invariant). One row per (method, channel, iteration 0..N) with levels (mean ± SE), `mean_delta`, *dz*, persona-bootstrap 95 % CI, Wilcoxon *p* and `p_holm` — here **Holm across iterations 0..N within (grader, method, channel)** (the paper's family; §1's `k_paired_channels` uses the other family, so *p* agrees cell-for-cell while `p_holm` need not). `k_channels_summary` counts, per (grader, method, channel), how many iterations clear Holm in each direction; `k_channels_grid` shows the three headline channels per grader plus the judge-invariant text channels.

In [ ]:
CKF = lookahead.channel_k_frames(CH, TM)         # {'channels', 'channels_text', 'channels_summary'}
KC, KT, KCS = CKF["channels"], CKF["channels_text"], CKF["channels_summary"]
print("channels:", KC.shape, "| text:", KT.shape, "| summary:", KCS.shape)
display(KC[KC.metric.isin(["MICI_OverPraise_rate", "MICI_OverPraise"]) & (KC.method == "PTO")].round(3))

for method in lookahead.METHODS:
    for j in JUDGES:
        d = KC[(KC.method == method) & (KC.judge == j)].drop(columns=["judge", "method"])
        exports.save_table(d, f"k_channels_{method.lower()}_{j}", caption=(
            f"**{method}, K=0 vs K=5 on the oracle-coded behaviour channels, grader = {JUDGE_TITLE[j]}.** MICI channels "
            "(per therapist turn `_rate`, per-session counts, MICI_Severity) are lower-better (+ = K=0 does MORE of a bad "
            "thing); MITI channels (`_per_turn`, counts, RtoQ, %CR, %MICO) are MI-consistent behaviours. Levels = mean +/- SE "
            f"over the 96 personas per arm; mean_delta, Cohen's dz, persona-bootstrap 95% CI (BOOT_SEED={BOOT_SEED}), "
            f"Wilcoxon p, p_holm. {lookahead.SIGN_NOTE} {lookahead.HOLM_NOTE} {lookahead.CENSOR_NOTE}"
            + support(CH[j], subject="no later scored state under this grader")
            + f" Paper fixture: k_contrast_headline_channels_{method.lower()}_{'primary' if j == PRIMARY else 'heldout'} "
            "(means/dz/p exact; CI bounds to bootstrap noise)."))
for method in lookahead.METHODS:
    d = KT[KT.method == method].drop(columns=["judge", "method"])
    exports.save_table(d, f"k_channels_text_{method.lower()}", caption=(
        f"**{method}, K=0 vs K=5 on the deterministic text channels (judge-invariant — computed from the transcripts, no "
        "grader involved).** conv_len = utterances per conversation; n_th_turns = therapist turns; mean_turn_len = "
        "characters per therapist turn; q_per_turn = literal '?' per therapist turn; loop = fraction of conversations with "
        f"a verbatim repeated therapist turn. All unvalenced (longer is not better). {lookahead.SIGN_NOTE} "
        f"{lookahead.HOLM_NOTE} {lookahead.CENSOR_NOTE}"
        + support(TM, subject="no transcripts for a later iteration")
        + f" Bootstrap CIs seeded with BOOT_SEED={BOOT_SEED} (means/dz/p exact w.r.t. the paper fixture "
        f"k_contrast_headline_channels_text_{method.lower()}; CI bounds differ by up to a few characters on mean_turn_len)."))
exports.save_table(KCS, "k_channels_summary", caption=(
    "**Per (grader, method, channel) summary of the behaviour-channel K contrast across iterations** (column `judge`; "
    f"the text channels carry judge = '{lookahead.TEXT_JUDGE_LABEL}'). Same columns as lookahead/reward's k_summary: "
    "n_sig_K0_higher / n_sig_K5_higher count iterations with Holm p<.05 and delta >0 / <0; the *_better columns flip the "
    "sign for lower-better channels (every MICI channel and count); mean_delta_iters1toN / mean_dz_iters1toN average the "
    "per-iteration paired deltas over TRAINED iterations only; base_delta / base_dz are the iteration-0 base-vs-base draw. "
    f"{lookahead.SIGN_NOTE} {lookahead.HOLM_NOTE} {lookahead.CENSOR_NOTE} n_iters records the support each row summarises. "
    "Paper fixture: k_contrast_headline_channels_summary."))

fig = plotting.k_channels_grid(KC, KT, palette=PAL)
exports.save_fig(fig, "k_channels_grid", caption=(
    "The K contrast on the behaviour channels: rows = graders (top gpt-4o-mini training oracle, bottom claude-haiku-4-5 "
    "held-out judge), columns = over-praise per turn, advice-without-permission per turn, MITI affirmations per turn "
    "(oracle-coded, per grader), and a last column of judge-invariant text channels (conv_len, mean_turn_len). Persona-"
    "paired K=0 - K=5 delta per iteration with 95% CI (PTO left-shifted, GRPO right-shifted; filled = Holm p<.05 across "
    "iterations 0..N within (grader, method, channel)). Sign: + => K=0 does MORE of it — MICI channels are lower-better, "
    "so + on the first two columns means K=0 is WORSE there. Iteration 0 = two independent base draws. Each method's "
    "series runs to its own last matched iteration, per grader — read the endpoint off the x axis."
    + support(CH[PRIMARY], subject="no later scored state under the training oracle")
    + " Paper fixture: k_contrast_headline_fig_channels."))
plt.show()

# ── ledger: the channel keys of the paper's k_contrast_headline.json (rubric keys live in lookahead/reward/k_numbers) ──
_empty = pd.DataFrame(columns=["judge", "method", "metric", "iteration"])
NUM_CH = {k: v for k, v in lookahead.lookahead_numbers(_empty, channels=KC, channels_text=KT, channels_summary=KCS).items()
          if k.startswith("channel") or k == "conventions"}
print(f"channel ledger: {len(NUM_CH)} keys")
exports.save_numbers("k_channels_numbers", NUM_CH, caption=(
    "Number ledger for the behaviour-channel K contrast: channel.<method>.<channel>.iter<n>.<judge> (the three headline "
    "channels of k_channels_grid, both graders), channel_text.<method>.<channel>.iter<n> (judge-invariant text channels) "
    "and channel_summary.<method>.<channel>.<judge>, each with its producing frame + row as `source`; `conventions` "
    "restates sign (+ => K=0 higher), pairing (persona_id, n = 96), Holm family, iteration 0 = two independent base draws "
    "and the support each key was read on (every key's iteration is the one its source row actually carries). The rubric "
    "keys of the same paper ledger (k_contrast_headline.json) live in lookahead/reward/tables/k_numbers.json."))

## 3 · Session shape — length, turns, verbosity, questions, loops (judge-invariant)  `[EVAL]`
**Purpose.** The ICLR paper reported that K=5 gives *shorter* conversations; this section replays that claim on the deterministic transcript metrics (`eda_analysis.replication` ← `session_shape_stability.py`): `session_shape` (persona-paired K0 − K5 per method × metric × iteration, **Holm within (method, metric) across iterations**), `length_endpoints` (base → final per arm), `length_kcontrast` (the paired K contrast at each method's LAST MATCHED iteration — read off the data, not fixed; on the completed grid both methods land at iteration 10), and `session_shape.png` (all four arms by iteration). No grader is involved anywhere here.

**`selection`** is the training-side counterpart, *copied not recomputed* from the tracked `arms/preference` tables (`update_lexical_push.md` + `generation_pool_means.md`): what lexical features each update pushed for (`w_len`, `w_question`, `w_affirm`, `w_overpraise`, Σ w·feature per group ± SE, on a shared scale for DPO's ±1 pair and GRPO's standardized advantages) versus what the policy generated (`pool_*`). Group-level, primary training oracle by construction, no persona pairing. ⚠ Until `arms/preference.ipynb` has rendered, the source falls back to the retired `results/{L0,L5}/tables/6_preference/gpt-4o-mini/` pair — the cell prints which it used and the caption records it.

In [ ]:
SHAPE = replication.session_shape_paired(TM)
LEVELS = replication.session_shape_levels(TM)
ENDP = replication.length_endpoints(LEVELS)                                       # conv_len, n_th_turns, mean_turn_len
ENDP_ALL = replication.length_endpoints(LEVELS, metrics=replication.SHAPE_METRICS)  # all five — the ledger's `levels`
KCON = replication.length_kcontrast(SHAPE)
print("shape:", SHAPE.shape, "| endpoints:", ENDP.shape, "| kcontrast:", KCON.shape)
display(SHAPE[SHAPE.metric == "conv_len"].round(3)); display(ENDP.round(3)); display(KCON.round(3))

# replication.CAPTIONS already carries the neutral support LEGEND (replication.CENSOR); the derived
# sentence is appended on top and is "" unless an arm is genuinely short.
exports.save_table(SHAPE, "session_shape", caption=(
    replication.CAPTIONS["shape"] + support(SHAPE, arm_col="method", label=False,
                                            subject="no later matched iteration in this frame")
    + " Same statistic as the paper fixture session_shape_stability_shape (means/dz/p exact; "
    f"bootstrap CIs at BOOT_SEED={BOOT_SEED} agree to a few characters on mean_turn_len / ~0.3 utterances on conv_len). "
    "This family's k_paired_channels (section 1) tests the same four length channels with Holm across channels within an "
    "iteration — same delta/dz/p, different p_holm scope."))
exports.save_table(ENDP, "length_endpoints", caption=(
    replication.CAPTIONS["length_endpoints"].replace("session_shape_stability_length_kcontrast.md", "length_kcontrast.md")
    + " Each arm's row is read at its own final scored iteration (the final_iteration column)."
    + support(ENDP, iter_col="final_iteration")
    + " Judge-free. Paper fixture: session_shape_stability_length_endpoints (exact)."))
exports.save_table(KCON, "length_kcontrast", caption=(
    replication.CAPTIONS["length_kcontrast"] + " Rows are pulled from session_shape at each method's last matched iteration "
    "(read off the data — the `iteration` column names it). Paper fixture: session_shape_stability_length_kcontrast "
    "(means/dz/p exact; CIs to bootstrap noise)."))

fig = plotting.shape_fig(LEVELS, arms=K_ARMS, palette=PAL)
exports.save_fig(fig, "session_shape", caption=(
    replication.CAPTIONS["fig_shape"] + " Left: conversation length (utterances); middle: therapist turn length (chars); "
    "right: questions per therapist turn — one line per arm, all four arms on one axis; iteration 0 = each arm's own base "
    "draw; each line runs to that arm's own last scored iteration."
    + support(LEVELS) + " Judge-free. Paper fixture: session_shape_stability_fig_shape."))
plt.show()

In [ ]:
# ── selection-level lexical push, COPIED from the tracked preference tables ──────────────────
_RESULTS_ROOT = exports.RESULTS_DIR                                                    # results/
NEW_SEL = os.path.join(_RESULTS_ROOT, "arms", "preference", "tables", "gpt-4o-mini")
if all(os.path.isfile(os.path.join(NEW_SEL, f)) for f in ("update_lexical_push.md", "generation_pool_means.md")):
    SEL_SRC, SEL_NOTE = NEW_SEL, "results/arms/preference/tables/gpt-4o-mini/ (all four arms in one table)"
else:
    SEL_SRC = replication.default_selection_dirs()
    SEL_NOTE = ("the RETIRED results/{L0,L5}/tables/6_preference/gpt-4o-mini/ pair (LA0 arms from L0, LA5 arms from L5) — "
                "arms/preference.ipynb had not rendered when this family was built; re-render after it lands")
    print("[note] arms/preference tables not on disk yet -> selection copied from the retired L0/L5 6_preference dirs")
print("selection source:", SEL_SRC)
SEL = replication.selection_table(SEL_SRC)
display(SEL.round(3))
exports.save_table(SEL, "selection", caption=(
    replication.CAPTIONS["selection"]
    + " Each arm's rows run to its own last recorded training iteration (the train_iter column)."
    + support(SEL, iter_col="train_iter", subject="no later training iteration recorded")
    + f" SOURCE for this render: {SEL_NOTE}. train_iter n samples from the iter-start "
    "policy, i.e. the eval set's model_iter_{n-1} (= policy_iteration). Paper "
    "fixture: session_shape_stability_selection (exact — copied numbers)."))

# ── ledger: the shape / length / selection part of the paper's session_shape_stability.json ──
NUM_SHAPE = replication.replication_numbers(shape=SHAPE, levels=ENDP_ALL, endpoints=ENDP, kcontrast=KCON, selection=SEL,
                                            table_prefix="tables/")
NUM_SHAPE = {k: {**v, "source": v["source"].replace("tables/shape.md", "tables/session_shape.md")} for k, v in NUM_SHAPE.items()}
print(f"shape ledger: {len(NUM_SHAPE)} keys")
# Named shape_numbers since 2026-08-26 (was replication_numbers, which COLLIDED with the
# different-content ledger of the same name in lookahead/replication; the archived paper's
# NUMBERS.md cites the old name).
exports.save_numbers("shape_numbers", NUM_SHAPE, caption=(
    "Number ledger for the session-shape part of this family (named shape_numbers 2026-08-26; was replication_numbers, "
    "a same-name different-content twin of the replication family's ledger): shape.<method>.<metric>.iter<n> (persona-paired "
    "K0 - K5, + => K=0 higher, n = 96), levels.<arm> (base -> final on all five shape metrics), "
    "length_endpoint.<method>.iter<n>.<metric> (the endpoint K contrast) and selection.<arm>.* (lexical push + pool means, "
    "copied from the preference tables). Sources name the table + row in this family. The dispersion / ceiling part of the "
    "same paper ledger (session_shape_stability.json sd.* / sd_bf.* / ceiling.*) lives in "
    "lookahead/replication/tables/replication_numbers.json."))

## 4 · Held-out instruments — WAI-SR subscales, patient change talk, the Q2 item profile, cooperation strata  `[EVAL]`
**Purpose.** The rubrics the training reward never saw, under both graders (`eda_analysis.instruments` ← `held_out_instruments.py`):
- **WAI-SR** — `wai_subscales` (Task / Goal / Bond levels + persona-paired gain over own base per arm × iteration; `bond_excess` = Bond − mean(Goal, Task)), `wai_kcontrast` (K0 − K5 per measure × matched iteration; a positive `bond_excess` delta means the K=0 arm's alliance gain is MORE bond-weighted), `wai_fig_data` + `wai.png` (gain at each arm's endpoint, plus a K=0 arm redrawn at its method's matched iteration whenever that differs from its own endpoint — derived, so on the completed grid there is no such extra bar).
- **PCT** — `pct_kcontrast`: K0 − K5 on the change-talk proportion (`PCT_ChangeProp` = the lake's `PCT`), the patient globals, and the utterance counts.
- **Q2 items** — `q2_items` (wide endpoint profile), `q2_items_long`, `q2_items_kcontrast` (17 items, Holm across items within (grader, method)); groups are a face-content reading, not a validated subscale.
- **Heterogeneity** — `hetero_kcontrast` + `hetero.png`: K0 − K5 WITHIN patient cooperation level (32 personas per stratum) at the matched endpoint and at each arm's own-oracle best; `share_*_ge_4.5` is the ceiling diagnostic for the Cooperative stratum.

Endpoints are read off the data (`endpoints` / `matched_endpoints`) — each method at its own matched pair, and the endpoint columns (`target_iter` / `iter_K0` / `iter_K5` / `@N`) record which iteration each row actually used. Sign `+ ⇒ K=0 higher`; paired on `persona_id`; graders side by side, never averaged.

In [ ]:
F = instruments.instrument_frames_by_judge(S.ARMS)      # {judge: {'wai_items','wai_subscales','wai_conv','q2_items','pct'}}
END = instruments.endpoints(SC[PRIMARY]); ME = instruments.matched_endpoints(END)
PARITY = {j: instruments.wai_subscale_parity(d["wai_conv"], d["wai_subscales"]) for j, d in F.items()}
print("endpoints:", END, "| matched:", ME, "| WAI parity:", PARITY)

WAI = instruments.wai_subscales(F, F)
WAIK = instruments.wai_kcontrast(F, F)
FIGW = instruments.wai_fig_data(F, end=END, matched_end=ME)
PCT = instruments.pct_kcontrast(F)
Q2 = instruments.q2_items(F, end=END, matched_end=ME)
HET = instruments.hetero_kcontrast(SC, end=END, matched_end=ME)
print("wai:", WAI.shape, "| wai_k:", WAIK.shape, "| fig:", FIGW.shape, "| pct:", PCT.shape,
      "| q2:", {k: v.shape for k, v in Q2.items()}, "| hetero:", HET.shape)
display(WAIK[(WAIK.measure == "bond_excess") & (WAIK.iteration.isin([5, 10]))].round(3))

# The support sentence every §4 caption interpolates: the module's neutral LEGEND plus whatever
# constants.support_note DERIVES from each grader's frame (nothing, while no arm is short).
SUPPORT = (instruments.CENSOR_NOTE
           + support(SC[PRIMARY], subject="no later scored state under the training oracle")
           + support(SC[HELDOUT], subject="no later scored state under the held-out judge"))
print("§4 support sentence:", SUPPORT)

_conv = (f"{instruments.SIGN_NOTE} {instruments.PAIR_NOTE} {SUPPORT} Iteration 0 = two independent base "
         f"draws (noise floor). Bootstrap CIs seeded with BOOT_SEED={BOOT_SEED} (means/dz/p exact w.r.t. the paper fixture; "
         "CI bounds to bootstrap noise). Column `judge` names the grader (gpt-4o-mini = training oracle, claude-haiku-4-5 = "
         "held-out); graders side by side, never averaged.")
exports.save_table(WAI, "wai_subscales", caption=(
    "**WAI-SR subscale levels + gain over own base**, per grader x arm x iteration. Subscales use the WAI-SR standard map "
    "(Task = items 1,2,10,12; Goal = 4,6,8,11; Bond = 3,5,7,9), asserted identical to the score lake's WAI_{Task,Goal,Bond}_Mean "
    f"columns under each grader (max |diff| = {max(p['max_abs_diff_items_vs_lake'] for p in PARITY.values()):.1e} over "
    f"{min(p['n_convs'] for p in PARITY.values())} conversations). `bond_excess` = Bond - mean(Goal, Task) per conversation, "
    "then averaged. `*_gain` = mean persona-paired difference vs the arm's OWN iteration-0 base (1-5 Likert points; "
    f"iteration-0 rows have gain 0 by construction). n = conversations (personas). {SUPPORT} Graders side "
    "by side (column `judge`), never averaged. Paper fixture: held_out_instruments_wai."))
exports.save_table(WAIK, "wai_kcontrast", caption=(
    "**Persona-paired K0-K5 contrast on the WAI-SR subscales**, per grader x method x measure x matched iteration. "
    "`bond_excess` = Bond - mean(Goal,Task) — a positive delta means the K=0 arm's alliance gain is MORE bond-weighted "
    "(relational) relative to its task/goal component than the K=5 arm's. mean_K0/mean_K5 = arm means on the paired "
    "personas; dz = mean/sd of paired deltas; 95% percentile-bootstrap CI (2000 draws); p = Wilcoxon signed-rank; p_holm = "
    f"Holm within (judge, method, measure) across iterations. {_conv} Paper fixture: held_out_instruments_wai_kcontrast."))
exports.save_table(FIGW, "wai_fig_data", caption=(
    "Data behind wai.png: persona-paired gain over own base per WAI-SR subscale at each arm's endpoint, plus a K=0 arm "
    "repeated at its method's matched iteration whenever that differs from its own endpoint (read off the data — the "
    "`iteration` column names what each row used), both graders. 95% percentile-bootstrap CI over "
    f"the paired deltas (BOOT_SEED={BOOT_SEED}). {instruments.PAIR_NOTE} Paper fixture: held_out_instruments_fig_wai_data."))
fig = plotting.wai_fig(FIGW, palette=PAL)
exports.save_fig(fig, "wai", caption=(
    "WAI-SR subscale gain over own base at each arm's endpoint (persona-paired, n = 96; whiskers = 95% bootstrap CI), one "
    "panel per grader (left gpt-4o-mini training oracle, right claude-haiku-4-5 held-out judge). Endpoints: each arm at "
    "its own last scored iteration, read off the data; where a method's MATCHED iteration differs from its K=0 arm's own "
    "endpoint, that K=0 arm is drawn again faded as the context for its K=5 sibling (no such faded bar when the arms end "
    f"together). K=5 bars hatched. {SUPPORT} Not a K contrast (read wai_kcontrast for that): every bar is an arm vs its "
    "OWN base. Paper fixture: held_out_instruments_fig_wai."))
plt.show()

exports.save_table(PCT, "pct_kcontrast", caption=(
    "**Persona-paired K0-K5 contrast on PCT (patient change talk) and its components**, per grader x method x metric x "
    "matched iteration. `PCT_ChangeProp` = CT/(CT+ST) is the score lake's `PCT` metric (higher = more change talk); "
    "`PCT_GlobalMean` = mean of the three 1-5 patient globals; the three utterance counts sum to `PCT_BehaviorTotal` "
    "(patient utterances coded — count-scale, so CI bounds there differ by up to ~0.2 from the paper's seed). "
    f"mean_K0/mean_K5 = arm means on the paired personas; dz = mean/sd of paired deltas; p = Wilcoxon; p_holm = Holm within "
    f"(judge, method, metric) across iterations. {_conv} Source: behavior.load_pct_behavior under each grader's partition of "
    "the score lake. Paper fixture: held_out_instruments_pct."))

exports.save_table(Q2["q2items"], "q2_items", caption=(
    "**Q2 item profile at the endpoint** (17 items of the Q2 relational-communication rubric, 1-5), per grader. "
    "`<arm>_gain@N` = persona-paired mean gain over the arm's OWN base at its final iteration N (the `@N` in each column "
    "header names that iteration); `base_mean(4 arms)` = mean of the four arms' independent base draws (descriptive). "
    "`<method>_K0-K5@N` = persona-paired K0-K5 contrast at the matched endpoint (`@N` names the iteration), + => K=0 higher; "
    "dz = mean/sd of paired deltas; p_holm = Holm across the 17 items within (judge, method). Groups are the face-content "
    "reading of constants.Q2_ITEM_GROUPS (analytical, not a validated subscale); items 3 and 10 = emotional self-disclosure, "
    f"items 1,2,3,10 = the self-disclosure group. {instruments.PAIR_NOTE} Graders side by side, never averaged. Paper "
    "fixture: held_out_instruments_q2items."))
exports.save_table(Q2["q2items_long"], "q2_items_long", caption=(
    "Long companion of q2_items: per (grader, arm, item) the persona-paired gain over the arm's own base at its endpoint "
    "(`target_iter`), with 95% percentile-bootstrap CI, dz and Wilcoxon p; `base`/`target` = arm means on the paired "
    f"personas. {instruments.PAIR_NOTE} {SUPPORT} CIs at BOOT_SEED={BOOT_SEED} (Likert scale; agree with the "
    "paper's to ~0.04). Paper fixture: held_out_instruments_q2items_long."))
exports.save_table(Q2["q2items_kcontrast"], "q2_items_kcontrast", caption=(
    "Per-item persona-paired K0-K5 contrast on the 17 Q2 items at each method's matched endpoint (the `iteration` column "
    f"names it per row). {SUPPORT} {instruments.SIGN_NOTE} {instruments.PAIR_NOTE} k_p_holm = Holm across the 17 "
    "items within (judge, method). Paper fixture: held_out_instruments_q2items_kcontrast."))

exports.save_table(HET, "hetero_kcontrast", caption=(
    "**K0-K5 contrast WITHIN patient cooperation level** (persona trait from the patient system prompt: High -> Cooperative, "
    "StartLowAndChangesToHigh -> Warms up, Low -> Resistant; 32 personas each), on Q1Q2 (the training reward, 1-5), MICI "
    "(MI-inconsistent behaviours per therapist turn; LOWER = better, so a positive delta means K=0 is WORSE) and PCT "
    f"(change-talk proportion; higher = better). {instruments.SIGN_NOTE} {instruments.PAIR_NOTE} target=matched_final: "
    "each method at its matched endpoint; target=own_best: each arm at its own-oracle best "
    "iteration (selected on the training oracle's Q1Q2 mean, reused for the held-out grader). Either way the iter_K0 / "
    f"iter_K5 columns name the iterations the row was read at. {SUPPORT} "
    "mean_K0/mean_K5 = subgroup arm means on the paired personas; dz = mean/sd of paired deltas; 95% percentile-bootstrap "
    "CI; p = Wilcoxon signed-rank; p_holm = Holm across the three cooperation subgroups within (judge, method, metric, "
    "target) — the 'All' row (all 96 personas) is a reference, outside the family. `share_*_ge_4.5` (Q1Q2 only) = fraction "
    "of that arm's subgroup conversations scoring >= 4.5, the ceiling diagnostic for the Cooperative stratum. Graders side "
    "by side, never averaged. Paper fixture: held_out_instruments_hetero."))
fig = plotting.hetero_fig(HET, palette=PAL)
exports.save_fig(fig, "hetero", caption=(
    "K0-K5 delta by patient cooperation stratum (Cooperative / Warms up / Resistant, 32 personas each) at the matched "
    "endpoint (each method at its own matched pair, read off the data — the iter_K0 / iter_K5 columns of "
    "hetero_kcontrast name it): rows Q1+Q2 (higher = better) and MICI (lower = better, "
    "so + = K=0 WORSE), columns = graders (left gpt-4o-mini training oracle, right claude-haiku-4-5 held-out judge); PTO vs "
    "GRPO bars coloured by the method's K=0 arm; whiskers = 95% bootstrap CI; * = Holm p<.05 across the three strata. "
    "Sign: + => K=0 higher; persona-paired. Paper fixture: held_out_instruments_fig_hetero."))
plt.show()

### 4b · The instruments ledger  `[EVAL]`
Every number the write-up may quote from the held-out instruments, as `{dotted.key: {value, source, note}}` → `tables/instruments_numbers.json` (`endpoints.*`, `wai.endpoint.*`, `wai.kcontrast.*`, `wai.fig_gain.*`, `pct.kcontrast.*`, `q2.item{1,2,3,10}.*`, `q2.groups.*`, `q2.kcontrast_summary.*`, `hetero.*`, `hetero.ceiling.*`, `wai.subscale_map_check.*`, `crosscheck.*`). The `source` strings are rewritten to this family's table names (the module defaults to the paper's `held_out_instruments_*` names).

In [ ]:
NUM_I = instruments.instruments_numbers(wai=WAI, wai_k=WAIK, fig_wai=FIGW, pct=PCT, q2=Q2, hetero=HET, scores_by_judge=SC,
                                        end=END, matched_end=ME, parity=PARITY, primary=PRIMARY,
                                        table_prefix="held_out_instruments")
_REMAP = [("tables/held_out_instruments_wai_kcontrast.md", "tables/wai_kcontrast.md"),
          ("tables/held_out_instruments_fig_wai_data.md", "tables/wai_fig_data.md"),
          ("figures/held_out_instruments_fig_wai.png", "figures/wai.png"),
          ("tables/held_out_instruments_wai.md", "tables/wai_subscales.md"),
          ("tables/held_out_instruments_pct.md", "tables/pct_kcontrast.md"),
          ("tables/held_out_instruments_q2items_long.md", "tables/q2_items_long.md"),
          ("tables/held_out_instruments_q2items_kcontrast.md", "tables/q2_items_kcontrast.md"),
          ("tables/held_out_instruments_q2items.md", "tables/q2_items.md"),
          ("tables/held_out_instruments_hetero.md", "tables/hetero_kcontrast.md")]
def _remap(s):
    for a, b in _REMAP:
        s = s.replace(a, b)
    return s
NUM_I = {k: {**v, "source": _remap(str(v.get("source", "")))} for k, v in NUM_I.items()}
print(f"instruments ledger: {len(NUM_I)} keys")
exports.save_numbers("instruments_numbers", NUM_I, caption=(
    "Number ledger for the held-out instruments (both graders): endpoints / matched endpoints / own-oracle best iterations, "
    "the WAI-SR endpoint levels + gains and their K contrast (wai.*), the change-talk K contrast (pct.*), the Q2 item "
    "profile (q2.*), the cooperation-stratum K contrast + ceiling shares (hetero.*), the WAI subscale-map parity check and "
    f"the two anchor crosschecks. Sign + => K=0 higher; paired on persona_id. {SUPPORT} Reproduces the paper's "
    "held_out_instruments.json key-for-key (values rounded to 3 decimals; CI bounds to bootstrap noise)."))

In [ ]:
exports.prune_orphan_captions(); exports.build_index()